In [4]:
"""
Test script for Wine Rating Prediction API
==========================================
"""

import requests
import json

# API endpoint
# BASE_URL = "http://0.0.0.0:9696"  # Local Docker
BASE_URL = "https://nuopel-wine-rating-api.hf.space"  # Hugging Face Spaces
# BASE_URL = "http://127.0.0.1:7860"  # Local Docker

def test_health_check():
    """Test the health check endpoint"""
    print("\n🔍 Testing health check...")
    try:
        response = requests.get(f"{BASE_URL}/")
        print(f"Status: {response.status_code}")
        if response.status_code == 200:
            print(f"Response: {json.dumps(response.json(), indent=2)}")
        else:
            print(f"Response text: {response.text}")
    except requests.exceptions.JSONDecodeError:
        print(f"⚠️ Non-JSON response received")
        print(f"Response text: {response.text[:200]}")  # First 200 chars


def test_single_prediction():
    """Test single wine prediction"""
    print("\n🍷 Testing single prediction...")

    # Example wine (typical Bordeaux characteristics)
    wine_data = {
        "vintage_year": 2018,
        "structure_acidity": 3.5,
        "structure_tannin": 3.2,
        "region": "Bordeaux"
    }

    try:
        response = requests.post(f"{BASE_URL}/predict", json=wine_data, timeout=30)
        print(f"Status: {response.status_code}")

        if response.status_code == 200:
            result = response.json()
            print(f"\n✨ Prediction Results:")
            print(f"   Predicted Rating: {result['predicted_rating']}")
            print(f"   Category: {result['rating_class']}")
        else:
            print(f"Error: {response.text}")
    except requests.exceptions.JSONDecodeError:
        print(f"⚠️ Non-JSON response received")
        print(f"Response text: {response.text[:200]}")


def test_batch_prediction():
    """Test batch prediction with multiple wines"""
    print("\n🍷🍷 Testing batch prediction...")

    wines = [
        {
            "vintage_year": 2018,
            "structure_acidity": 3.5,
            "structure_tannin": 3.2,
            "region": "vin-de-pays-vignobles-de-france"
        },
        {
            "vintage_year": 2020,
            "structure_acidity": 1.2,
            "structure_tannin": 2.8,
            "region": "haut-medoc"
        },
        {
            "vintage_year": 1983,
            "structure_acidity": 1.2,
            "structure_tannin": 3.8,
            "region": "pauillac"
        }
    ]

    try:
        response = requests.post(f"{BASE_URL}/predict_batch", json=wines, timeout=30)
        print(f"Status: {response.status_code}")

        if response.status_code == 200:
            results = response.json()
            print(f"\n✨ Batch Predictions ({len(results)} wines):")
            for i, result in enumerate(results, 1):
                print(f"\n   Wine {i}:")
                print(f"      Rating: {result['predicted_rating']}")
                print(f"      Category: {result['rating_class']}")
        else:
            print(f"Error: {response.text}")
    except requests.exceptions.JSONDecodeError:
        print(f"⚠️ Non-JSON response received")
        print(f"Response text: {response.text[:200]}")


def test_invalid_input():
    """Test API with invalid input"""
    print("\n❌ Testing invalid input handling...")

    # Missing required field
    invalid_wine = {
        "vintage_year": 2018,
        "structure_acidity": 3.5
        # Missing structure_tannin and region
    }

    response = requests.post(f"{BASE_URL}/predict", json=invalid_wine, timeout=30)
    print(f"Status: {response.status_code}")
    print("Expected 422 (Validation Error)")
    if response.status_code == 422:
        print(f"✅ Validation working correctly")
    print(f"Response: {response.text[:200]}")


if __name__ == "__main__":
    print("=" * 60)
    print("🧪 Wine Rating API Test Suite")
    print(f"Testing: {BASE_URL}")
    print("=" * 60)

    try:
        # Run all tests
        test_health_check()
        test_single_prediction()
        test_batch_prediction()
        test_invalid_input()

        print("\n" + "=" * 60)
        print("✅ All tests completed!")
        print("=" * 60)

    except requests.exceptions.ConnectionError as e:
        print("\n❌ Error: Could not connect to API")
        print(f"Make sure the API is running on {BASE_URL}")
        print(f"Error details: {e}")
    except Exception as e:
        print(f"\n❌ Unexpected error: {e}")

🧪 Wine Rating API Test Suite
Testing: https://nuopel-wine-rating-api.hf.space

🔍 Testing health check...
Status: 200
Response: {
  "status": "healthy",
  "service": "Wine Rating Prediction API",
  "model_loaded": true,
  "encoder_loaded": true
}

🍷 Testing single prediction...
Status: 200

✨ Prediction Results:
   Predicted Rating: 3.92
   Category: Good

🍷🍷 Testing batch prediction...
Status: 200

✨ Batch Predictions (3 wines):

   Wine 1:
      Rating: 3.87
      Category: Good

   Wine 2:
      Rating: 3.8
      Category: Good

   Wine 3:
      Rating: 4.0
      Category: Very Good

❌ Testing invalid input handling...
Status: 422
Expected 422 (Validation Error)
✅ Validation working correctly
Response: {"detail":[{"type":"missing","loc":["body","structure_tannin"],"msg":"Field required","input":{"vintage_year":2018,"structure_acidity":3.5}},{"type":"missing","loc":["body","region"],"msg":"Field requ

✅ All tests completed!
